In [2]:
import nltk
# nltk.download('averaged_perceptron_tagger_eng')
# nltk.download('punt_tab')

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
from pathlib import Path
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from tqdm import tqdm
import contractions
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk import pos_tag

In [3]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
# Initialize stemmer/lemmatizer (run once)
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Step 2: Load the speeches

In [4]:
base_path_alvaro = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Project")

base_path = base_path_alvaro # change according to user

In [5]:
final_df = pd.read_csv(base_path / "Final_df.csv")
final_df = final_df.drop(columns=["Speech", "number_sentences", "number_tokens","matched_climate_keywords", "speeches_for_keyword_search"])
final_df

,Session,Year,ISO-Code,Income Level,cleaned_speeches_postagging_expanded,contains_climate_keyword
0,45,1990,AFG,1,allow first sir congratulate unanimous electio...,False
1,45,1990,AGO,2,first would like congratulate sir election pre...,False
2,45,1990,ALB,2,special pleasure speak session general assembl...,False
3,45,1990,ARE,4,president behalf delegation united arab emirat...,False
4,45,1990,ARG,2,president general assembly fortyfifth session ...,False
...,...,...,...,...,...,...
6452,79,2024,WSM,2,excellency extend congratulation excellency ph...,True
6453,79,2024,YEM,1,lady gentleman happy coincidence address today...,True
6454,79,2024,ZAF,3,president session general assembly philemon ya...,True
6455,79,2024,ZMB,2,lady gentleman congratulate excellency assumpt...,True


In [ ]:
# Climate keyword according to the literature
climate_keywords = ["climate change", "global warming", "global warm","cap and trade", "paris accord", "emissions trading", "global average temperature", 
                      "kyoto protocol", "changing climate" ,"climate resilience","climate decay", "carbon dioxide", "carbon-dioxide","climate politics", 
                      "framework convention climate change", "bali roadmap", "bali action plan", 
                      "greenhouse gas", "greenhouse-gas","greenhouse effect", "climate mitigation", "climate action", "emissions", "temperature", "extreme weather", 
                      "global environmental change", "global environment", "global environmental" ,"climate variability", "low carbon", "renewable energy", 
                      "carbon emission", "climate pollutant", "climate pollutants", "carbon tax", "carbon footprint", "carbon neutrality", "net-zero", 
                      "net zero","net-zero","climate crisis", "climate summit", "climate catastrophe", "climate justice", "climate emergency", "climate funding",
                      "climate fund", "climate financing", "climate finance","climate peace", "climate agreement", "climate security", "climate ambition", 
                      "climate issue", "climate impact", "climate conference", "climate event", "climate challenge", "climate trust", "climate negotiation", 
                      "climate catastrophe", "climate risk", "climate goal", "climate change-related", "climate regime", "climate resilient", "climate policy", 
                      "carbon market", "carbon sink", "green climate", "green economy", "emission reduction", "emissions reduction", "carbon neutral", 
                      "ozone layer", "unfccc", "emissions trading scheme", "ghg", "ghge", "co2", "co2 emission", "ipcc", "decarbonisation", "decarbonization"]

In [ ]:
climate_keywords_dict = {
    # 1. Climate Science & Impacts
    "science_impacts": [
        "climate change", "global warm", "global warming", "global average temperature",
        "climate variability", "extreme weather", "climate impact",
        "greenhouse effect", "temperature", "climate catastrophe",
        "climate risk", "ozone layer", "global environmental change",
        "global environmental", "global environment", 
        "climate change-related", "climate risk", "changing climate",
        "climate decay"
    ],

    # 2. Policy & Agreements
    "policy_agreements": [
        "paris accord", "kyoto protocol", "unfccc", "climate policy",
        "framework convention on climate change", "climate agreement",
        "climate regime", "climate negotiation", "climate ambition",
        "climate security", "bali roadmap", "ipcc", "bali action plan"
    ],

    # 3. Carbon & Emissions
    "carbon_emissions": [
        "carbon dioxide", "co2", "carbon emission", "carbon tax",
        "carbon footprint", "carbon neutrality", "carbon neutral",
        "carbon market", "carbon sink", "low carbon", "emissions",
        "emission reduction", "emissions reduction", "ghg", "ghge",
        "climate pollutant", "climate pollutants", "carbon neutral",
        "greenhouse gas", "decarbonisation", "decarbonization", 
        "greenhouse-gas"
    ],

    # 4. Climate Action & Solutions
    "action_solutions": [
        "climate action", "climate mitigation", "renewable energy",
        "climate resilience", "climate resilient", "green economy",
        "climate funding", "climate fund", "climate financing", "climate finance",
        "net-zero", "net zero", "green climate", "cap and trade",
        "emissions trading scheme", "emissions trading"
    ],

    # 5. Sociopolitical Climate Issues
    "sociopolitical": [
        "climate justice", "climate emergency", "climate crisis",
        "climate summit", "climate politics", "climate peace",
        "climate challenge", "climate goal", "climate issue",
        "climate event", "climate conference", "climate trust"
    ]
}
    